# 🌾 Validación de Modelos en Agronomía

Este laboratorio interactivo demuestra los diferentes métodos para validar modelos predictivos, usando datos reales de **Evapotranspiración de Referencia (ET₀)**.

In [ ]:
%pip install plotly pandas requests scikit-learn nbformat -q

In [2]:
import pandas as pd
import requests
import plotly.graph_objects as go
from sklearn.model_selection import TimeSeriesSplit, GroupKFold
from plotly.subplots import make_subplots
import numpy as np

## 1. Adquisición de Datos (Open-Meteo)
Descargaremos 3 años de datos diarios de ET₀ para una zona agrícola. En este caso, usaremos coordenadas aproximadas de la zona de Talca para tener valores climáticos representativos.

In [3]:
# Coordenadas agrícolas (Ej: Cercanías de Talca)
lat = -35.42
lon = -71.66

url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2021-01-01&end_date=2023-12-31&daily=et0_fao_evapotranspiration&timezone=America%2FSantiago"
response = requests.get(url)
data = response.json()

df = pd.DataFrame({
    'Fecha': pd.to_datetime(data['daily']['time']),
    'ET0': data['daily']['et0_fao_evapotranspiration']
}).dropna()

df.set_index('Fecha', inplace=True)
print(f"Total de días descargados: {len(df)}")
df.head()

Total de días descargados: 1095


,ET0
Fecha,
2021-01-01,7.42
2021-01-02,6.58
2021-01-03,8.31
2021-01-04,8.29
2021-01-05,7.42


## 2. Visualización Base
Observemos el comportamiento estacional de la Evapotranspiración.

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['ET0'], mode='lines', name='ET₀ (mm/día)', line=dict(color='#2ca02c')))
fig.update_layout(title='Evapotranspiración Histórica (ET₀)', xaxis_title='Fecha', yaxis_title='ET₀ (mm/día)', template='plotly_white')
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 3. Validación Simple (Train-Test Split)
El enfoque más básico: Cortamos la serie y usamos el pasado para entrenar y el futuro para probar.

In [ ]:
# Modifica el tamaño de prueba aquí (ej. 0.2 para dejar el 20% final como prueba)
porcentaje_prueba = 0.2
corte_idx = int(len(df) * (1 - porcentaje_prueba))

train = df.iloc[:corte_idx]
test = df.iloc[corte_idx:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train['ET0'], mode='lines', name='Entrenamiento (Train)', line=dict(color='#1f77b4')))
fig.add_trace(go.Scatter(x=test.index, y=test['ET0'], mode='lines', name='Prueba (Test)', line=dict(color='#ff7f0e')))
fig.update_layout(title='Train-Test Split Simple', template='plotly_white')
fig.show()

## 4. Validación en Series Temporales (Time Series CV)
En agronomía el tiempo importa. Aquí vemos cómo la 'ventana de entrenamiento' crece cronológicamente.

In [ ]:
# Cambia el número de divisiones (folds)
n_splits = 4
tscv = TimeSeriesSplit(n_splits=n_splits)

fig = make_subplots(rows=n_splits, cols=1, shared_xaxes=True)

for i, (train_index, test_index) in enumerate(tscv.split(df)):
    train_cv = df.iloc[train_index]
    test_cv = df.iloc[test_index]
    
    fig.add_trace(go.Scatter(x=train_cv.index, y=train_cv['ET0'], mode='lines', name=f'Train {i+1}', line=dict(color='#1f77b4')), row=i+1, col=1)
    fig.add_trace(go.Scatter(x=test_cv.index, y=test_cv['ET0'], mode='lines', name=f'Test {i+1}', line=dict(color='#ff7f0e')), row=i+1, col=1)

fig.update_layout(title_text='Validación de Series Temporales (Ventana Expansiva)', height=800, template='plotly_white', showlegend=False)
fig.show()

### 4.1 Predicciones en cada fold
Hasta aquí solo vimos *cómo* se parte la serie. Ahora entrenamos un modelo en cada ventana de entrenamiento y predecimos su bloque de prueba.

- **Predictores:** la ET₀ de los días anteriores (rezagos de 1, 2, 3 y 7 días) y la época del año (seno y coseno del día). Todos se conocen *antes* del día que se predice.
- **Línea base:** persistencia, es decir, "mañana será igual que hoy". Un modelo solo aporta si le gana a esta regla trivial.
- **Horizonte:** un día hacia adelante. Para predecir cada día de prueba se usa la ET₀ real del día anterior.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Predictores: solo información disponible ANTES del día a predecir
df_modelo = pd.DataFrame({'ET0': df['ET0']})
for lag in (1, 2, 3, 7):
    df_modelo[f'lag_{lag}'] = df['ET0'].shift(lag)
dia_del_año = df.index.dayofyear
df_modelo['sin_anual'] = np.sin(2 * np.pi * dia_del_año / 365.25)
df_modelo['cos_anual'] = np.cos(2 * np.pi * dia_del_año / 365.25)
df_modelo = df_modelo.dropna()

X = df_modelo.drop(columns='ET0')
y = df_modelo['ET0']

resultados = []
for i, (train_index, test_index) in enumerate(tscv.split(X)):
    modelo = RandomForestRegressor(n_estimators=200, random_state=42)
    modelo.fit(X.iloc[train_index], y.iloc[train_index])

    y_test = y.iloc[test_index]
    pred = pd.Series(modelo.predict(X.iloc[test_index]), index=y_test.index)
    persistencia = X['lag_1'].iloc[test_index]

    resultados.append({
        'Fold': i + 1,
        'Días train': len(train_index),
        'Días test': len(test_index),
        'MAE modelo': mean_absolute_error(y_test, pred),
        'MAE persistencia': mean_absolute_error(y_test, persistencia),
    })

print(pd.DataFrame(resultados).set_index('Fold').round(3))

# Al salir del ciclo, y_test, pred y persistencia corresponden al último fold
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test.index, y=y_test, mode='lines', name='Real', line=dict(color='#ff7f0e')))
fig.add_trace(go.Scatter(x=pred.index, y=pred, mode='lines', name='Predicción (Random Forest)', line=dict(color='#1f77b4')))
fig.add_trace(go.Scatter(x=persistencia.index, y=persistencia, mode='lines', name='Persistencia', line=dict(color='#7f7f7f', dash='dot'), visible='legendonly'))
fig.update_layout(title=f'Predicción vs. valor real en el último fold (fold {n_splits})', xaxis_title='Fecha', yaxis_title='ET₀ (mm/día)', template='plotly_white')
fig.show()

## 5. GroupKFold (Leave-Group-Out CV)
Ideal cuando tenemos múltiples sensores, plantas o cuarteles. Evita que el modelo 'memorice' las condiciones biológicas de un individuo/cuartel específico.

In [ ]:
# Simularemos que estos datos provienen de 3 cuarteles distintos
np.random.seed(42)
df_simulado = pd.DataFrame({
    'Fecha': df.index.tolist() * 3,
    'ET0': df['ET0'].tolist() + (df['ET0'] * 1.1).tolist() + (df['ET0'] * 0.9).tolist(),
    'Cuartel': ['Norte']*len(df) + ['Sur']*len(df) + ['Este']*len(df)
})

gkf = GroupKFold(n_splits=3)
grupos = df_simulado['Cuartel']

# Mostramos textualmente qué cuartel se usa para prueba en cada iteración
for i, (train_idx, test_idx) in enumerate(gkf.split(df_simulado, groups=grupos)):
    cuartel_prueba = df_simulado.iloc[test_idx]['Cuartel'].unique()[0]
    cuarteles_train = df_simulado.iloc[train_idx]['Cuartel'].unique()
    print(f"Iteración {i+1}:")
    print(f"  🌳 Entrenando con cuarteles: {cuarteles_train}")
    print(f"  🎯 Probando con cuartel excluido: ['{cuartel_prueba}']\n")